In [2]:
import anthropic
import base64
import httpx
from pathlib import Path

In [3]:
def encode_image(image_path: str) -> tuple[str, str]:
    path = Path(image_path)
    suffix = path.suffix.lower()
    media_type_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    media_type = media_type_map.get(suffix, "image/jpeg")
    with open(image_path, "rb") as f:
        image_data = base64.standard_b64encode(f.read()).decode("utf-8")
    return image_data, media_type
    
        

In [4]:
def caption_from_file(image_path: str, prompt: str = None)-> str:
    client = anthropic.Anthropic()
    image_data, media_type = encode_image(image_path)

    user_prompt = prompt or "Describe the image in detail. Include objects, colors, mood, and any text visible."

    message = client.messages.create(
        model = "claude-opus-4-5",
        max_tokens = 1024,
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": user_prompt,
                    },
                ],
            }
        ],
    )
    return message.content[0].text

In [5]:
def batch_caption(image_paths: list[str]) -> dict[str,str]:
    results = {}
    for path in image_paths:
        print(f"Processing: {path}")
        results[path] = caption_from_file(path)
    return results

In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

print("Loading model... (may take a few minutes on first run)")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
print("Model loaded")


d:\tensorFlow\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model... (may take a few minutes on first run)


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 473/473 [00:02<00:00, 186.28it/s, Materializing param=vision_model.post_layernorm.weight]                                       
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT t

Model loaded


In [7]:
def caption_from_file(image_path: str, prompt: str = None) -> str:
    image = Image.open(image_path).convert("RGB")
    
    if prompt:
        # conditional captioning with a prompt
        inputs = processor(image, prompt, return_tensors="pt")
    else:
        # unconditional captioning
        inputs = processor(image, return_tensors="pt")
    
    output = model.generate(**inputs, max_new_tokens=100)
    return processor.decode(output[0], skip_special_tokens=True)

In [8]:
import os

image_path = r"C:\Users\Ehsan\Pictures\your_image.jpg"  # ← change this

# verify file exists first
if not os.path.exists(image_path):
    print(" File not found! Check the path.")
else:
    caption = caption_from_file(image_path)
    print("\n=== Caption ===")
    print(caption)


=== Caption ===
a man in a suit and tie with his arms crossed
